# Causalite-02 — Exiger un estimand : l'identification causale avant le chiffre

[← Causalite-01](Causalite-01-Calcul-Do-Python.ipynb) | [↑ Causalité](README.md) | [Causalite-03 →](Causalite-03-DoWhy-Contrefactuel-Individuel-Python.ipynb)

> Renumerotation #17421 : ce carnet etait publie sous le nom DoWhy-1 ; les renvois internes de la serie citent desormais le numero courant.

**Serie** : Inference causale avec DoWhy (Probas, Python)
**Prerequis** : [PyMC-05-Causal-Inference](../PyMC/PyMC-05-Causal-Inference.ipynb) (l'echelle de Pearl,
le do-calcul sur un SCM connu) et le
[notebook-pont Do-Calculus-Bridge](Causalite-01-Calcul-Do-Python.ipynb) (le
criteres backdoor et front-door demontres une fois chacun sur leurs cas).
**Outil** : [DoWhy](https://py-why.github.io/dowhy/) 0.14 — le vrai outil d'identification causale, pas une reimplementation.

Le vocabulaire causal est facile a emprunter et difficile a honorer. Ecrire `do(X)` devant une
distribution ne fait pas une intervention ; ce qui la fait, c'est un **estimand** : l'identification
EXPLICITE de la quantite causale visee a partir de ce qui est observable, sous un graphe ASSUME.

Le pont a montre UNE fois comment dowhy identifie (backdoor sur college/earnings, frontdoor sur
genotype/tar). Ce notebook change d'exigence : sur un seul cas, tenu du generateur a la refutation,
chaque etape doit dire quelle hypothese elle suppose et ce qui arrive au chiffre quand l'hypothese
saute :

1. Quel graphe supposons-nous — et que suppose-t-il, qu'exclut-il ?
2. Quel estimand en decoule — backdoor, frontdoor, variable instrumentale — nomme, pas implicite ?
3. Que devient le chiffre si le graphe change — la sensibilite mesuree, pas commentee ?


## 1. Le cas propre — reeducation post-operatoire

Un hopital deploye un **nouveau protocole de reeducation intensive** (`T`) et mesure la **mobilite
recuperee a 3 mois** (`Y`). Le service de biostatistiques fournit 3 000 dossiers. Deux autres
variables vivent dans chaque dossier :

- `Z` — **severite initiale** observee (score clinique pre-operatoire) ;
- `M` — **adherence au programme** (nombre de seances effectuees, binarisee).

Et le protocole n'est pas assigne au hasard : **les cas severes sont orientes plus souvent** vers le
nouveau protocole, et la severite pousse aussi la recuperation (`+3.0` points de mobilite par unite
de severite) : un patient plus atteint au depart dispose de plus de marge de progression sur trois
mois. Deux fleches partent donc de `Z`, et c'est tout le probleme.

Sur CE cas, la difference entre association et intervention est concrete, pas theorique :

- **Association** : « comparer la mobilite des patients qui ont recu le protocole a celle de ceux
  qui ne l'ont pas recu ». Les deux groupes different precisement parce que les cas severes sont
  sur-representes chez les traites — le chiffre melange l'effet du protocole et l'effet de la
  severite.
- **Intervention** : « administrer le protocole a TOUS les patients, a severite egale, et mesurer
  l'ecart avec personne ne le recevant ». C'est `P(Y | do(T=1)) - P(Y | do(T=0))`, une quantite que
  l'observation seule ne donne jamais sans hypothese.

Pour que la lecon soit verifiable, ce notebook SIMULE les dossiers : le generateur est le monde,
donc la verite de l'intervention est accessible par re-simulation — un juge que la vraie vie ne
fournit pas, et que DoWhy ne verra jamais : l'outil ne recoit que le tableau de donnees et le
graphe que nous lui donnons.


In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
N = 3000

# Le monde (verite du generateur, invisible de DoWhy) :
#   Z (severite)     -> T, -> Y     confondeur observe
#   T (protocole)    -> M           traitement
#   M (adherence)    -> Y           mediateur
Z = rng.binomial(1, 0.5, N)                          # severite initiale (0: legere, 1: severe)
T = rng.binomial(1, 1 / (1 + np.exp(-(1.2 * Z - 0.8))))  # assignation depend de la severite
M = rng.binomial(1, 1 / (1 + np.exp(-(1.5 * T + 0.5))))  # adherence depend du protocole
Y = 2.0 * T + 1.5 * M + 3.0 * Z + rng.normal(0, 1, N)    # mobilite recuperee

dossiers = pd.DataFrame({"Z": Z, "T": T, "M": M, "Y": Y})
dossiers.head()

,Z,T,M,Y
0,1,1,1,6.340874
1,1,0,1,4.648793
2,1,0,0,2.553809
3,0,1,1,3.442216
4,0,1,1,4.489245


### Lecture de la sortie

La sortie montre les cinq premières lignes du tableau `dossiers`, avec ses quatre colonnes `Z`, `T`, `M`, `Y` : la ligne 0 rassemble `Z=1`, `T=1`, `M=1` et `Y=6.340874`, et la ligne 2 un cas sévère non traité (`Z=1`, `T=0`) pour `Y=2.553809`. C'est sur ce tableau que toutes les estimations du notebook vont être calculées.

## 2. La verite de l'intervention, par re-simulation

Le generateur permet ce que le terrain interdit : figer `T` par une intervention (`do`) et
re-simuler le monde. L'effet **total** du protocole se decompose en :

- un **effet direct** `T -> Y` de `2.0` points de mobilite ;
- un **effet indirect** `T -> M -> Y` : le protocole pousse l'adherence, l'adherence pousse la
  recuperation (`1.5` points par patient adherent).

La simulation Monte-Carlo a grande echelle donne le juge de paix auquel chaque estimand sera
confronte. En pratique ce chiffre n'existe pas — c'est justement pourquoi l'identification
(causalite) ne se reduit pas a l'estimation (statistique).

In [2]:
def simuler_monde(t_fixe, n=200_000, seed=999):
    """Re-simule le monde avec T IMPOSE par intervention (do)."""
    g = np.random.default_rng(seed)
    z = g.binomial(1, 0.5, n)
    m = g.binomial(1, 1 / (1 + np.exp(-(1.5 * t_fixe + 0.5))), size=n)
    return 2.0 * t_fixe + 1.5 * m + 3.0 * z + g.normal(0, 1, n)

verite_totale = simuler_monde(1).mean() - simuler_monde(0).mean()
print(f"Effet total vrai  do(T=1) - do(T=0) : {verite_totale:.3f} points de mobilite")

Effet total vrai  do(T=1) - do(T=0) : 2.390 points de mobilite


### Lecture de la sortie

La re-simulation par intervention fixe l'effet total vrai du protocole à **2.390 points de mobilité** : c'est la différence `do(T=1) - do(T=0)` estimée sur 200 000 réplications, le juge de paix auquel chaque estimand des sections suivantes sera comparé.

## 3. L'association naive — et son biais, chiffre

La difference de moyennes `P(Y | T=1) - P(Y | T=0)` est ce que produit un `groupby` sans hypothese
causale. Elle repond a une question d'association : « a quoi ressemblent les patients traites,
comparés aux autres ? » — pas a la question d'intervention qui interesse l'hopital.

In [3]:
naif = dossiers[dossiers["T"] == 1]["Y"].mean() - dossiers[dossiers["T"] == 0]["Y"].mean()
print(f"Association naive P(Y|T=1) - P(Y|T=0) : {naif:.3f}")
print(f"Biais de confusion (naif - verite)    : {naif - verite_totale:+.3f}"
      f"  ({100 * (naif - verite_totale) / verite_totale:+.0f} % de l'effet vrai)")
print()
print("Le chiffre observe surestime l'effet du protocole : les cas severes (recuperation plus")
print("elevee, +3.0 points par unite) sont sur-representes chez les traites, donc le groupe")
print("traite part avec un avantage qui n'appartient pas au protocole. Le biais est POSITIF")
print("parce que l'arc Z -> Y est positif : changer son signe changerait le sens du biais.")


Association naive P(Y|T=1) - P(Y|T=0) : 3.274
Biais de confusion (naif - verite)    : +0.884  (+37 % de l'effet vrai)

Le chiffre observe surestime l'effet du protocole : les cas severes (recuperation plus
elevee, +3.0 points par unite) sont sur-representes chez les traites, donc le groupe
traite part avec un avantage qui n'appartient pas au protocole. Le biais est POSITIF
parce que l'arc Z -> Y est positif : changer son signe changerait le sens du biais.


### Lecture de la sortie

L'association naïve affiche **3.274**, soit un biais de confusion de **+0.884** par rapport à la vérité simulée (2.390) : le simple écart de moyennes `P(Y|T=1) - P(Y|T=0)` surestime l'effet du protocole, parce que les cas sévères sont sur-représentés chez les traités.

## 4. Le graphe assume — puis l'estimand, nomme

DoWhy ne devine rien : `CausalModel` recoit le tableau ET un graphe DOT que NOUS ecrivons. Ce
graphe est une prise de position, pas un resultat :

- il **suppose** que la severite `Z` cause a la fois l'assignation `T` et la recuperation `Y`
  (chemin `T <- Z -> Y`, la porte arriere) ;
- il **suppose** que l'adherence `M` est un mediateur (`T -> M -> Y`), pas un confondeur ;
- il **exclut** tout autre chemin de dependence (pas d'arc `M -> T`, pas de latent commun a `T`
  et `Y` — hypothese forte que la section 7 mettra a l'epreuve).

`identify_effect` traduit alors le graphe en estimand : la stratégie employée est **nommee**
(backdoor), et la formule d'ajustement est explicite — « DoWhy a renvoye un nombre » n'est pas un
estimand.

In [4]:
from dowhy import CausalModel

graphe = 'digraph { Z -> T; Z -> Y; T -> M; M -> Y; }'

modele = CausalModel(
    data=dossiers, treatment="T", outcome="Y", graph=graphe,
)

estimand = modele.identify_effect(proceed_when_unidentifiable=True)
print(estimand)
print()
print("Set d'ajustement backdoor :", estimand.get_backdoor_variables())

Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estimand name: backdoor
Estimand expression:
 d          
────(E[Y|Z])
d[T]        
Estimand assumption 1, Unconfoundedness: If U→{T} and U→Y then P(Y|T,Z,U) = P(Y|T,Z)

### Estimand : 2
Estimand name: iv
No such variable(s) found!

### Estimand : 3
Estimand name: frontdoor
Estimand expression:
 ⎡ d       d       ⎤
E⎢────(Y)⋅────([M])⎥
 ⎣d[M]    d[T]     ⎦
Estimand assumption 1, Full-mediation: M intercepts (blocks) all directed paths from T to Y.
Estimand assumption 2, First-stage-unconfoundedness: If U→{T} and U→{M} then P(M|T,U) = P(M|T)
Estimand assumption 3, Second-stage-unconfoundedness: If U→{M} and U→Y then P(Y|M, T, U) = P(Y|M, T)

### Estimand : 4
Estimand name: general_adjustment
Estimand expression:
 d          
────(E[Y|Z])
d[T]        
Estimand assumption 1, Unconfoundedness: If U→{T} and U→Y then P(Y|T,Z,U) = P(Y|T,Z)


Set d'ajustement backdoor : ['Z']


L'estimand backdoor dit : ajuster sur `Z` suffit pour fermer toutes les portes arrieres. Le
mediateur `M`, lui, ne doit PAS etre ajuste — ajuster un mediateur amputerait l'effet de sa partie
indirecte. Reste a estimer la formule sur les donnees.

La sortie le confirme : l'estimand retourné est de type `NONPARAMETRIC_ATE`, nommé **backdoor**, avec le jeu d'ajustement `['Z']` ; la piste instrumentale échoue (« No such variable(s) found! »), et une formule frontdoor via `M` est proposée en troisième position.

In [5]:
effet_backdoor = modele.estimate_effect(
    estimand, method_name="backdoor.linear_regression",
)

comparaison = pd.DataFrame({
    "Quantite": ["Verite (do, par simulation)", "Association naive", "Backdoor ajustee sur Z"],
    "Estimation": [verite_totale, naif, effet_backdoor.value],
})
comparaison["Ecart a la verite"] = comparaison["Estimation"] - verite_totale
comparaison.round(3)

,Quantite,Estimation,Ecart a la verite
0,"Verite (do, par simulation)",2.390,0.000
1,Association naive,3.274,0.884
2,Backdoor ajustee sur Z,2.382,-0.008


### Lecture de la sortie

Le tableau comparatif résume le parcours : la vérité simulée vaut **2.390**, l'association naïve **3.274** (écart +0.884), et l'estimation backdoor ajustée sur `Z` **2.382**, soit **-0.008** par rapport à la vérité. L'ajustement a retiré quasiment tout le biais de confusion sans toucher au médiateur `M`.

## 5. La sensibilite au graphe — mesuree, pas commentee

Un estimand qui ne bouge pas quand une hypothese causale fausse est retire est suspect ; un qui
explose est informatif. Les quatre graphes candidats ci-dessous different d'UNE seule hypothese
chacun, et chaque chiffre est compare au juge de la section 2 :

| Graphe candidat | Hypothese retenue |
|---|---|
| **A — correct** | le graphe du generateur |
| **B — sans `Z -> T`** | « la severite n'influence pas l'assignation » : `Z` n'est plus un confondeur, aucun ajustement requis |
| **C — sans `Z -> Y`** | « la severite n'influence pas la recuperation » : l'autre negation de la porte arriere |
| **D — fleche inversee** | « l'adherence cause l'assignation » (`M -> T` au lieu de `T -> M`) : plus AUCUN chemin dirige de `T` vers `Y` |

In [6]:
variantes = {
    "A correct":    "digraph { Z -> T; Z -> Y; T -> M; M -> Y; }",
    "B sans Z->T":  "digraph { Z -> Y; T -> M; M -> Y; }",
    "C sans Z->Y":  "digraph { Z -> T; T -> M; M -> Y; }",
    "D fleche inv": "digraph { Z -> T; Z -> Y; M -> T; M -> Y; }",
}

def set_ajustement(estimand):
    """Set backdoor d'un estimand ; distingue les trois cas None."""
    if "No directed path" in str(estimand):
        return "effet declare nul"
    try:
        return sorted(estimand.get_backdoor_variables()) or "(aucun)"
    except TypeError:  # backdoor_variables est None : aucune porte arriere declaree
        return "(aucun)"

lignes = []
for nom, dot in variantes.items():
    m = CausalModel(data=dossiers, treatment="T", outcome="Y", graph=dot)
    e = m.identify_effect(proceed_when_unidentifiable=True)
    r = m.estimate_effect(e, method_name="backdoor.linear_regression")
    lignes.append({
        "Graphe": nom,
        "Ajustement": set_ajustement(e),
        "Estime": round(r.value, 3),
        "Ecart a la verite": round(r.value - verite_totale, 3),
    })

sensibilite = pd.DataFrame(lignes)
sensibilite

,Graphe,Ajustement,Estime,Ecart a la verite
0,A correct,[Z],2.382,-0.008
1,B sans Z->T,(aucun),2.789,0.400
2,C sans Z->Y,(aucun),3.274,0.884
3,D fleche inv,effet declare nul,0.000,-2.390


Lecture du tableau :

- **B et C portent le MÊME estimand** : la colonne `Ajustement` affiche `(aucun)` pour les deux.
  Retirer `Z -> T` (B) supprime la porte arrière — il n'y a plus rien à fermer ; retirer `Z -> Y`
  (C) laisse `Z -> T` mais supprime tout chemin de `Z` vers `Y`, donc la porte arrière est
  déclarée absente elle aussi. Deux graphes faux différemment, une même réponse d'identification.
- **Ce qui sépare B de C n'est donc pas l'identification : c'est l'estimateur.** Dans B, l'arc
  `Z -> Y` survit alors que `Z` n'est plus un confondeur ; DoWhy le promeut en *effect modifier*
  et régresse `Y ~ T + T*Z` — plus un simple écart de moyennes, mais une pente en `T` qui varie
  avec `Z`, moyennée sur l'échantillon — un rôle qu'aucune colonne du tableau ne montre, et que la
  cellule de falsification ci-dessous imprime. La regle est mecanique, pas arbitraire :
  sans `effect_modifiers` explicites, `estimate_effect` retient les ancetres de `Y`, en
  ecarte ceux de `T` et les noeuds des chemins diriges `T -> Y`
  (`CausalModel.estimate_effect` -> `graph.get_effect_modifiers`, DoWhy 0.14). En A comme
  en C, `Z` est ancetre de `T` : ecarte. En B, la fleche `Z -> T` a disparu : `Z` survit,
  et l'interaction `T*Z` entre dans la regression. Dans C,
  plus aucun arc de `Z` vers `Y`, donc plus de modifier : la régression se réduit à `Y ~ T` et
  rend l'association naïve. **Expérience de falsification** : la cellule suivante rejoue le graphe B en interdisant la
  promotion (`effect_modifiers=[]`) — l'estimé retombe exactement sur le chiffre de C, et les
  modificateurs promus passent de `['Z']` à `[]`. Même estimand, même nombre dès que l'estimateur
  est remis à plat.
  L'écart entre B et C est donc un artefact de conditionnement résiduel, pas un « reste de biais »
  que le graphe B aurait correctement fermé.
- **D** est la catastrophe la plus instructive : inverser la flèche `T -> M` en `M -> T` supprime
  tout chemin dirigé de `T` vers `Y` — l'identification conclut « effet causal NUL » et l'estimé
  rend exactement 0.000. L'hypothèse fausse ne brouille pas le chiffre, elle DÉCIDE du chiffre :
  croire que l'adhérence cause l'assignation, c'est déclarer que le protocole ne peut rien.

Trois signatures distinctes, et une seule est une histoire d'identification : casser la flèche
d'ENTRÉE du confondeur (B) laisse un chiffre plus proche de la vérité, mais par un effet
d'estimateur et non parce qu'un biais résiduel aurait été correctement fermé ; nier la flèche de
SORTIE (C) rend l'estimation égale à l'association naïve, tout le biais de confusion revient ;
se tromper de SENS pour une seule flèche (D) écrase l'effet à zéro. Aucune des trois ne crie
toute seule — c'est la comparaison au graphe correct, et au juge de la section 2, qui les révèle.
La leçon de méthode compte autant que le résultat : **une colonne qui bouge entre deux graphes
peut bouger pour une raison d'estimation, sans que l'hypothèse d'identification ait changé.**
Se fier à la proximité du juge sans lire l'estimand aurait fait préférer un modèle dont la
ressemblance est un accident.


In [7]:
# Falsification de la lecture : l'ecart B/C est-il identification ou estimation ?
# La boucle laisse m et e sur la DERNIERE variante (D) : on reconstruit le modele de B
# explicitement, sinon estimate_effect estimerait D et rendrait 0.000.
mB = CausalModel(data=dossiers, treatment="T", outcome="Y", graph=variantes["B sans Z->T"])
eB = mB.identify_effect(proceed_when_unidentifiable=True)
rB_defaut = mB.estimate_effect(eB, method_name="backdoor.linear_regression")
rB_plat = mB.estimate_effect(eB, method_name="backdoor.linear_regression", effect_modifiers=[])

# dowhy promeut Z en effect modifier sur B : design Y ~ T + T*Z (3 coefficients).
# A plat, design Y ~ T (2 coefficients) -- le meme que C, donc le meme estimé.
for etiquette, r in [("B defaut", rB_defaut), ("B modifiers=[]", rB_plat)]:
    print(f"{etiquette:15s}: {r.value:+.3f}  |  modificateurs promus : "
          f"{list(r.estimator._effect_modifier_names)}  |  coefficients du design : "
          f"{len(r.estimator.model.params)}")

c_ref = float(sensibilite.set_index("Graphe").loc["C sans Z->Y", "Estime"])
print(f"{'C (tableau)':15s}: {c_ref:+.3f}  <- B remis a plat retombe sur ce chiffre")


B defaut       : +2.789  |  modificateurs promus : ['Z']  |  coefficients du design : 3
B modifiers=[] : +3.274  |  modificateurs promus : []  |  coefficients du design : 2
C (tableau)    : +3.274  <- B remis a plat retombe sur ce chiffre


## 6. sensibilite causale vs robustesse statistique

La section 5 fait varier le **graphe** (hypothese causale). DoWhy fournit aussi des **refuters**
statistiques — placebo, ajout d'un confondeur aleatoire, sous-echantillonnage — qui testent la
robustesse de l'ESTIMATION aux donnees (le pont Do-Calculus-Bridge les utilise deja comme
garde-fou). Les deux familles repondent a des questions differentes, et l'experience ci-dessous
le prouve : on passe les TROIS refuters sur le **graphe C faux** (celui qui nie `Z -> Y` et rend
l'estimation egale au naive biaise de +0.9).

In [8]:
def passer_refuters(modele, estimand, effet, etiquette):
    print(f"=== {etiquette}")
    for refuter in ["placebo_treatment_refuter", "random_common_cause",
                    "data_subset_refuter"]:
        rep = modele.refute_estimate(estimand, effet, method_name=refuter,
                                     random_seed=7)
        print(f"  {refuter:28s} : {rep.new_effect:+.3f}")

# Graphe correct : les refuters confirment la solidite statistique.
passer_refuters(modele, estimand, effet_backdoor, "Graphe correct (A)")

# Graphe C FAUX : l'estimation vaut le naive (biais +0.88)...
mC = CausalModel(data=dossiers, treatment="T", outcome="Y", graph=variantes["C sans Z->Y"])
eC = mC.identify_effect(proceed_when_unidentifiable=True)
rC = mC.estimate_effect(eC, method_name="backdoor.linear_regression")
print(f"  estimation sur graphe C           : {rC.value:+.3f} (verite {verite_totale:.3f})")

# ...et les refuters statistiques ne s'en apercoivent PAS.
passer_refuters(mC, eC, rC, "Graphe faux (C) — refuters quand meme verts")

=== Graphe correct (A)


  placebo_treatment_refuter    : +0.005


  random_common_cause          : +2.382


  data_subset_refuter          : +2.384
  estimation sur graphe C           : +3.274 (verite 2.390)
=== Graphe faux (C) — refuters quand meme verts


  placebo_treatment_refuter    : -0.002


  random_common_cause          : +3.274


  data_subset_refuter          : +3.275


Le graphe C passe ses trois refuters haut la main : placebo proche de zero, estimateur stable
au confondeur aleatoire et au sous-echantillonnage. Vert partout — et faux de +0.88. C'est la
preuve experimentale que la robustesse statistique ne detecte pas une faute causale : les refuters
interrogent les DONNEES, jamais le GRAPHE. Seule la sensibilite de la section 5 (relire une
hypothese du graphe et mesurer l'ecart) attaque la bonne couche.

## 7. Un second cas pour l'exercice : application de revision

Un editeur mesure l'effet d'une **application de revision** (`X`) sur le **score final** (`Y`).
Le terrain (`U` — motivation, niveau initial) est **latent** : il cause a la fois l'adoption de
l'application et le score, et n'apparait dans AUCUN fichier de donnees. Le temps d'etude (`M`,
heures declarees) est, lui, observe. Le generateur ci-dessous fixe la verite — l'identification
de ce cas est l'objet de l'**exercice 2** : la porte arriere est fermee (confondeur latent), une
autre strategie est necessaire. La demonstration pas-a-pas du frontdoor vit au §5 du
[pont Do-Calculus-Bridge](Causalite-01-Calcul-Do-Python.ipynb) sur le cas
genotype/goudron ; ici, c'est a vous de la reconstruire sur un cas neuf.

In [9]:
rng2 = np.random.default_rng(42)
N2 = 4000
U2 = rng2.normal(0, 1, N2)                    # terrain latent (NON observe)
X2 = 0.8 * U2 + rng2.normal(0, 1, N2)         # usage de l'application
M2 = 1.2 * X2 + rng2.normal(0, 0.8, N2)       # heures d'etude (mediateur observe)
Y2 = 1.5 * M2 + 0.6 * U2 + rng2.normal(0, 1, N2)  # score final

scores = pd.DataFrame({"X": X2, "M": M2, "Y": Y2})   # U volontairement absent

def simuler_terrain(x_fixe, n=200_000, seed=1234):
    g = np.random.default_rng(seed)
    u = g.normal(0, 1, n)
    m = 1.2 * x_fixe + g.normal(0, 0.8, n)
    return 1.5 * m + 0.6 * u + g.normal(0, 1, n)

verite_scores = simuler_terrain(1.0).mean() - simuler_terrain(0.0).mean()

med = scores["X"].median()
haut, bas = scores[scores["X"] > med], scores[scores["X"] <= med]
naif_scores = haut["Y"].mean() - bas["Y"].mean()
print(f"Effet total vrai  do(X=1) - do(X=0)   : {verite_scores:.4f} points")
print(f"Association naive (mediane split)    : {naif_scores:.4f} — biais {naif_scores - verite_scores:+.4f}")

# Le split median ne compare PAS deux valeurs voisines de X : les deux groupes sont
# separes de ~2 unites, alors que do(X=1) - do(X=0) vaut UNE unite. Sans le mesurer,
# tout l'ecart serait porte au compte du confondeur latent.
ecart_x = haut["X"].mean() - bas["X"].mean()
x, y = scores["X"].to_numpy(), scores["Y"].to_numpy()
pente_naive = np.cov(x, y)[0, 1] / np.var(x, ddof=1)

# La confusion se mesure avec un estimateur INDEPENDANT des moyennes de groupe : la
# pente naive. La definir comme le residu de l'ecart la rendrait infaillible par
# construction -- elle se refermerait sur n'importe quel generateur, meme non lineaire,
# et le partage ne dirait plus rien. Le troisieme terme porte justement ce que le
# modele lineaire ne capte pas.
part_portee = verite_scores * (ecart_x - 1.0)
part_confusion = (pente_naive - verite_scores) * ecart_x
part_reste = (naif_scores - verite_scores) - part_portee - part_confusion
print(f"Ecart moyen de X entre les 2 groupes : {ecart_x:.4f} unites")
print(f"Ecart observe moins l'effet vrai     : {naif_scores - verite_scores:+.4f}")
print(f"  part due a la PORTEE du contraste  : {part_portee:+.4f}")
print(f"  part due a la CONFUSION par U      : {part_confusion:+.4f}   ((pente - effet vrai) x ecart)")
print(f"  reste non capte par le lineaire    : {part_reste:+.4f}   (ecart aux moyennes - somme des deux)")
print(f"Pente naive par unite de X (Y ~ X)   : {pente_naive:.4f} — biais {pente_naive - verite_scores:+.4f}")
scores.head()


Effet total vrai  do(X=1) - do(X=0)   : 1.8000 points
Association naive (mediane split)    : 4.2336 — biais +2.4336
Ecart moyen de X entre les 2 groupes : 2.0601 unites
Ecart observe moins l'effet vrai     : +2.4336
  part due a la PORTEE du contraste  : +1.9082
  part due a la CONFUSION par U      : +0.5287   ((pente - effet vrai) x ecart)
  reste non capte par le lineaire    : -0.0033   (ecart aux moyennes - somme des deux)
Pente naive par unite de X (Y ~ X)   : 2.0567 — biais +0.2567


,X,M,Y
0,0.496978,0.863256,1.167926
1,0.063231,1.064952,1.033674
2,0.873682,2.021082,2.241138
3,2.991283,3.228775,4.019747
4,-0.131041,-0.002035,-0.991376


### Lecture de la sortie

La sortie fixe la vérité à **1.8000 point** de score par unité de `X`, contre **4.2336** pour l'association naïve (séparation à la médiane) — un écart de **+2.4336**. Cet écart ne se lit pas d'un bloc : le split médian compare deux groupes séparés de **2.0601 unités de `X`**, là où `do(X=1) - do(X=0)` vaut une unité.

La décomposition imprimée le sépare en trois termes, dont les deux premiers viennent d'**estimateurs indépendants** — et c'est ce qui la rend falsifiable :

- **+1.9082** de portée du contraste : l'effet vrai d'une unité, ré-extrapolé sur les 2.0601 unités qui séparent les deux groupes ;
- **+0.5287** de confusion par `U`, lu sur la **pente naïve de la régression**, qui ne regarde pas les moyennes de groupe. La définir comme le résidu de l'écart aurait produit une identité infaillible par construction, qui se refermerait sur n'importe quel générateur, même non linéaire ;
- **-0.0033** de reste : l'écart entre la différence de moyennes du split et ce que la pente prédit sur 2.06 unités. Il est quasi nul ici parce que le générateur est **linéaire en `X`**. Sur un générateur non linéaire, c'est ce terme qui grossirait — et le partage en deux parts n'aurait alors plus de sens, ce que le carnet enseigne précisément à repérer ailleurs.

Le partage suppose donc deux hypothèses, toutes deux explicites : la **linéarité en `X`**, et l'**extrapolation** de l'effet d'une unité sur la largeur du contraste. Ramenée à une unité de `X`, la pente naïve vaut **2.0567**, soit un biais de **+0.2567** seulement. Le terrain latent `U` gonfle donc le chiffre observé, mais il n'en explique pas le quart : le reste est une comparaison entre deux populations trop éloignées sur `X`. Le tableau `scores` n'expose que `X`, `M`, `Y` — `U` reste invisible de DoWhy, ce qui est tout l'enjeu de l'exercice 2.


## 8. Exercices

Trois exercices pour que l'exigence d'estimand devienne un reflexe. Chaque stub s'execute sans
erreur (`print` de consigne) — completer, pas remplacer.


### Exercice 1 — Le graphe assume

Une chaine de retail hésite entre deux leviers : une **campagne publicitaire** (`P`) et une
**baisse de prix** (`B`) ; la cible est le **chiffre de ventes** (`V`). Indices du terrain : les
campagnes sont lancees en periode de forte **saisonnalite** (`S`, observee), qui porte aussi les
ventes ; la campagne augmente la **notoriete** (`N`, observee) qui porte les ventes ; le budget
promo (`B`) est decide par les memes equipes qui observent les ventes faiblir. Ecrire le graphe
DOT suppose (variables observees seulement), puis lister par ecrit : (a) les chemins de porte
arriere vers `P`, (b) ce que le graphe EXCLUT explicitement, (c) le set d'ajustement attendu.

In [10]:
# Exercice 1 : le graphe assume
# Etape 1 : declarer le graphe DOT (variables observees : S, P, B, N, V).
# Etape 2 : le passer a CausalModel (treatment='P', outcome='V') et identifier.
# Indice : la saisonnalite S est un confondeur observe ; N est un mediateur, pas un confondeur.
# Attendu : l'estimand doit NOMMER la strategie et le set d'ajustement.

graphe_retail = None  # TODO etudiant : "digraph { ... }"

print("Exercice a completer : ecrire graphe_retail (DOT), puis CausalModel + identify_effect,")
print("et noter (a) les portes arrieres, (b) les exclusions du graphe, (c) le set d'ajustement.")

Exercice a completer : ecrire graphe_retail (DOT), puis CausalModel + identify_effect,
et noter (a) les portes arrieres, (b) les exclusions du graphe, (c) le set d'ajustement.


### Exercice 2 — L'estimand (et son NOM)

Sur le cas de la section 7 (application de revision, terrain latent `U`, mediateur observe `M`) :
construire le `CausalModel` — le graphe doit declarer `U` comme `[latent]` — puis identifier,
estimer, et **nommer la strategie d'identification** employee. Le verifier contre
`verite_scores` : un ecart faible confirme que la strategie ferme la bonne porte.

In [11]:
# Exercice 2 : l'estimand
# Etape 1 : graphe DOT avec U [latent] -> {X, Y} ; X -> M -> Y.
# Etape 2 : CausalModel(treatment='X', outcome='Y', graph=...) puis identify_effect.
# Etape 3 : estimer avec la methode adequate et comparer a verite_scores.
# Indice : backdoor est FERME (U latent) — quelle porte reste-t-il ? Le nommer explicitement.

print("Exercice a completer : identifier + estimer l'effet de X sur Y, et NOMMER la strategie.")
print("Verdict attendu : estimand ~ verite_scores =", round(verite_scores, 3))

Exercice a completer : identifier + estimer l'effet de X sur Y, et NOMMER la strategie.
Verdict attendu : estimand ~ verite_scores = 1.8


### Exercice 3 — La réfutation : le tableau de sensibilite

Reprendre le cas principal (sections 1-5) et produire le tableau de sensibilite de VOTRE graphe :
au moins trois hypotheses retorques (arêtes retirees ou ajoutees, differentes de B/C/D), chacune
avec son estime et son ecart a la verite. Un tableau, pas un commentaire — et une phrase de
lecture : quelle hypothese, si elle est fausse, fait le plus mal au chiffre ?

In [12]:
# Exercice 3 : le tableau de sensibilite
# Etape 1 : choisir 3 hypotheses a retirer (ex. retirer M -> Y ; ajouter Z -> M ; ajouter T -> Z).
# Etape 2 : pour chaque variante, identifier + estimer, collecter (estime, ecart a la verite).
# Etape 3 : rendre un DataFrame colonnes = Hypothese | Estime | Ecart a la verite.
# Indice : simuler les variants dans un dict {nom: dot} comme la section 5.

lignes_ex3 = []  # TODO etudiant : remplir avec les trois variantes choisies

print("Exercice a completer : produire le tableau (DataFrame) de sensibilite a 3+ hypotheses,")
print("puis une phrase : quelle hypothese fausse degrade le plus l'estimation ?")

Exercice a completer : produire le tableau (DataFrame) de sensibilite a 3+ hypotheses,
puis une phrase : quelle hypothese fausse degrade le plus l'estimation ?


## 9. Synthese

| Question | Reponse du notebook | Ou |
|---|---|---|
| Association ou intervention ? | `P(Y\|T=1)-P(Y\|T=0)` melange severite et protocole ; seul `do(T)` isole le protocole — sur CE cas, +0.9 d'ecart | sections 2-3 |
| Quel estimand ? | backdoor, set {Z}, NOMME par `identify_effect` — et le mediateur `M` explicitement NON ajuste | section 4 |
| Le graphe est-il solide ? | sensibilite mesuree : nier `Z -> Y` ramene tout le biais de confusion (+0.88) ; traiter `M` comme confondeur ecrase l'effet (estimation 0.000, ecart -2.39) | section 5 |
| Et la variance ? | refuters statistiques (placebo, subset) — robustesse des donnees, qui ne protege PAS d'un graphe faux | section 6 |
| Porte fermee ? | frontdoor via le mediateur observe, quand le confondeur est latent | exercice 2 |

L'estimand n'est pas une formalite avant le chiffre : c'est le contrat qui dit quelle quantite le
chiffre estime, sous quelles hypotheses, et lesquelles de ces hypotheses — si elles sautent —
emportent la conclusion. Exiger l'estimand, c'est exiger que ces trois reponses soient ecrites AVANT
que le nombre n'ait l'air d'etre une preuve.


## À retenir

| Leçon | Mesure dans ce carnet |
|---|---|
| L'identification se **nomme** avant l'estimation : `dowhy` produit quatre estimands distincts sur le même graphe | `Estimand name: backdoor` |
| L'effet total de vérité se mesure par simulation d'intervention | `Effet total vrai do(T=1) - do(T=0) : 2.390 points de mobilite` |
| L'association brute surestime l'effet du protocole de plus d'un tiers | `Biais de confusion (naif - verite) : +0.884` |
| L'ajustement backdoor sur Z récupère la vérité à l'erreur d'échantillon près | `Backdoor ajustee sur Z 2.382 -0.008` |
| Retirer une flèche du graphe dégrade l'estimation sans le signaler | `C sans Z->Y (aucun) 3.274 0.884` |
| Inverser la flèche fait sortir l'effet à zéro : c'est l'estimation qui le déclare nul | `D fleche inv effet declare nul 0.000 -2.390` |
| Les réfutations ne testent pas le graphe : elles restent vertes sur le mauvais graphe | `estimation sur graphe C : +3.274 (verite 2.390)` |
| Le placebo ne distingue pas les deux graphes | `placebo_treatment_refuter : -0.002` |
| Sur le second cas, la pente naïve surestime aussi l'effet | `Pente naive par unite de X (Y ~ X) : 2.0567 — biais +0.2567` |
| Le gros du biais vient de la **portée** du contraste, pas de la confusion | `part due a la PORTEE du contraste : +1.9082` |
| La part liée au confondeur caché est bien plus faible | `part due a la CONFUSION par U : +0.5287` |
| Le résidu non linéaire est négligeable | `reste non capte par le lineaire : -0.0033` |

## Vérifiez votre compréhension

1. L'ajustement sur Z rend-il l'estimation exacte ?

<details><summary>Réponse</summary>

Presque : `Backdoor ajustee sur Z 2.382 -0.008` ne s'écarte que de 0,008 de la vérité `Effet total vrai do(T=1) - do(T=0) : 2.390 points de mobilite`. Mais cette justesse est **conditionnelle au graphe** : changez le graphe, le tableau change (`C sans Z->Y (aucun) 3.274 0.884`). Le carnet ne conclut donc pas que l'ajustement suffit en général — il conclut que *sur ce graphe-là* l'ajustement est le bon. La question restante, « le graphe est-il le vrai ? », n'est pas tranchée par l'estimation.

</details>

2. Les tests de robustesse valident-ils le graphe ?

<details><summary>Réponse</summary>

Non. Sur le graphe faux C, les trois contrôles passent (`placebo_treatment_refuter : -0.002`) alors que l'estimation vaut `estimation sur graphe C : +3.274 (verite 2.390)`. Les refuters vérifient la **stabilité** de l'estimateur, pas la validité du modèle causal : un estimateur faussé mais stable les satisfait tous. L'inversion du sens causal le montre mieux encore — `D fleche inv effet declare nul 0.000 -2.390` : un effet nul passe aussi inaperçu.

</details>

3. Le biais de l'association observée est-il entièrement de la confusion ?

<details><summary>Réponse</summary>

Non, et le carnet le décompose. Sur le cas suivant, `part due a la PORTEE du contraste : +1.9082` en explique l'essentiel, `part due a la CONFUSION par U : +0.5287` est la fraction imputable à la cause non mesurée, et `reste non capte par le lineaire : -0.0033` reste marginal. Un découpage médian modifie d'abord l'ampleur de la comparaison : la régression naïve `Pente naive par unite de X (Y ~ X) : 2.0567 — biais +0.2567` n'explique donc pas l'écart « à cause de la confusion ».

</details>


## Pour aller plus loin

- [Index de la sous-série Decision Theory / Causalité](README.md) — la place de ce carnet et de ses voisins dans la sous-série.
- [Index de la série Probas](../../README.md) — le fil complet de la série (Probas, Decision Theory, Applications).
- Le carnet `Causalite-01-Calcul-Do-Python` pose les trois règles du do-calculus et les critères *backdoor* / *front-door* : celui-ci les met en application sur un cas complet, estimand nommé.
- Le carnet `Causalite-03-DoWhy-Contrefactuel-Individuel-Python` descend d'un cran : là où ici l'effet est un nombre moyen, il devient un contrefactuel par individu.